In [ ]:
!pip install langchain openai chromadb tiktoken faiss-cpu

In [ ]:
!pip install langsmith langchain-openai

In [ ]:
!pip install streamlit gradio

In [ ]:
!pip install --upgrade langchain langchain-openai openai chromadb tiktoken faiss-cpu langsmith

In [23]:
import os
import pandas as pd
import glob

from openai import OpenAI

from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import Chroma, FAISS
from langchain.chat_models import ChatOpenAI
from langchain.document_loaders import CSVLoader
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationalRetrievalChain
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain.text_splitter import RecursiveCharacterTextSplitter


In [24]:
#setting up
from dotenv import load_dotenv

load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

MODEL = "gpt-4.1-nano"

langsmith_api_key = os.getenv('LANGSMITH_API_KEY')
if langsmith_api_key:
    print(f"langsmith API Key exists and begins {langsmith_api_key[:8]}")
else:
    print("langsmith API Key not set")


openai = OpenAI()
MODEL = "gpt-4.1-nano"

OpenAI API Key exists and begins sk-proj-
langsmith API Key exists and begins lsv2_pt_


In [25]:
LANGSMITH_TRACING=True
LANGSMITH_ENDPOINT="https://api.smith.langchain.com"
LANGSMITH_PROJECT="Client_360_Positions_Summarize"

In [26]:
# ========== Step 1: Load Excel files ==========
csv_files = glob.glob("Data/ai_generated_data/upload_dataset/*.csv")
if not csv_files:
    raise FileNotFoundError("No csv files found in Data/ai_generated_data/. Drop your files there or change the path.")

entire_knowledge_base = ""

for file_path in csv_files:
    with open(file_path, 'r', encoding='utf-8') as f:
        entire_knowledge_base += f.read()
        entire_knowledge_base += "\n\n"

dataframes = [pd.read_csv(file) for file in csv_files]
combined_df = pd.concat(dataframes, ignore_index=True).fillna('')
print("Combined DataFrame shape:", combined_df.shape)

combined_df["page_content"] = combined_df.astype(str).agg(" | ".join, axis=1)

combined_df.head()

Combined DataFrame shape: (1000, 14)


,client_id,le_id,sub_prof_id,orig_customer_name,customer_name,cust_seg,cust_seg_desc,status_code,onboard_date,client_size,inc_num,mth_end,src,cty,page_content
0,C100000,LE001,SP001,Pacific Exports Pte Ltd,Pacific Exports Pte Ltd,A,Local Corporate,Active,04-03-2011,Small,DEFAULT,2015-06-30,SCI,SG,C100000 | LE001 | SP001 | Pacific Exports Pte ...
1,C100001,LE001,SP001,Evergreen Commodities Pte Ltd,Evergreen Commodities Pte Ltd,ME,Medium enterprises,Active,20-04-2007,Medium,DEFAULT,2022-03-26,SCI,SG,C100001 | LE001 | SP001 | Evergreen Commoditie...
2,C100002,LE001,SP001,Sunrise Logistics Pte Ltd,Sunrise Logistics Pte Ltd,50,Private Banking,Active,18-09-2005,Large,DEFAULT,2010-09-02,SCI,SG,C100002 | LE001 | SP001 | Sunrise Logistics Pt...
3,C100003,LE001,SP001,Sunrise Logistics Pte Ltd,Sunrise Logistics Pte Ltd,30,Business Clients,Active,02-05-2016,Small,DEFAULT,2023-07-03,SCI,SG,C100003 | LE001 | SP001 | Sunrise Logistics Pt...
4,C100004,LE001,SP001,SIMANGGANG SAWMILL COMPANY LTD SINGAPORE,SIMANGGANG SAWMILL COMPANY LTD SINGAPORE,K,Global corporate,Active,24-09-2020,Small,DEFAULT,2022-03-23,SCI,SG,C100004 | LE001 | SP001 | SIMANGGANG SAWMILL C...


In [27]:
# How many tokens in all the documents?

import tiktoken

encoding = tiktoken.encoding_for_model(MODEL)
tokens = encoding.encode(entire_knowledge_base)
token_count = len(tokens)
print(f"Total tokens for {MODEL}: {token_count:,}")

Total tokens for gpt-4.1-nano: 53,149


In [28]:
# ========== Step 2: Convert to Documents ==========
from langchain.document_loaders import DataFrameLoader
loader = DataFrameLoader(combined_df[["page_content"]], page_content_column="page_content")
documents = loader.load()

documents


[Document(metadata={}, page_content='C100000 | LE001 | SP001 | Pacific Exports Pte Ltd | Pacific Exports Pte Ltd | A | Local Corporate | Active | 04-03-2011 | Small | DEFAULT | 2015-06-30 | SCI | SG'),
 Document(metadata={}, page_content='C100001 | LE001 | SP001 | Evergreen Commodities Pte Ltd | Evergreen Commodities Pte Ltd | ME | Medium enterprises | Active | 20-04-2007 | Medium | DEFAULT | 2022-03-26 | SCI | SG'),
 Document(metadata={}, page_content='C100002 | LE001 | SP001 | Sunrise Logistics Pte Ltd | Sunrise Logistics Pte Ltd | 50 | Private Banking | Active | 18-09-2005 | Large | DEFAULT | 2010-09-02 | SCI | SG'),
 Document(metadata={}, page_content='C100003 | LE001 | SP001 | Sunrise Logistics Pte Ltd | Sunrise Logistics Pte Ltd | 30 | Business Clients | Active | 02-05-2016 | Small | DEFAULT | 2023-07-03 | SCI | SG'),
 Document(metadata={}, page_content='C100004 | LE001 | SP001 | SIMANGGANG SAWMILL COMPANY LTD SINGAPORE | SIMANGGANG SAWMILL COMPANY LTD SINGAPORE | K | Global corp

In [29]:
# ========== Step 3: Split documents ==========
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
docs = splitter.split_documents(documents)

In [ ]:
!pip install faiss-cpu

In [30]:
# ========== Step 4: Embeddings + Vector DB ==========
embeddings = OpenAIEmbeddings()
vectordb = FAISS.from_documents(docs, embeddings)
vectordb.save_local("faiss_client360_index")  # persist for reuse

In [31]:
# ========== Step 5: Create LLM + Memory + Conversational RAG Chain ==========
llm = ChatOpenAI(model_name=MODEL, temperature=0)  # change model_name as you prefer

memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

qa_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=vectordb.as_retriever(search_kwargs={"k": 3}),
    memory=memory,
    verbose=False
)

In [ ]:
!pip install streamlit

In [ ]:
def make_chain(vectordb):
    llm = ChatOpenAI(model_name=MODEL, temperature=0)
    memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)
    chain = ConversationalRetrievalChain.from_llm(
        llm=llm,
        retriever=vectordb.as_retriever(search_kwargs={"k": 3}),
        memory=memory,
        verbose=False,
    )
    return chain

[Document(metadata={}, page_content='C100000 | LE001 | SP001 | Pacific Exports Pte Ltd | Pacific Exports Pte Ltd | A | Local Corporate | Active | 04-03-2011 | Small | DEFAULT | 2015-06-30 | SCI | SG'),
 Document(metadata={}, page_content='C100001 | LE001 | SP001 | Evergreen Commodities Pte Ltd | Evergreen Commodities Pte Ltd | ME | Medium enterprises | Active | 20-04-2007 | Medium | DEFAULT | 2022-03-26 | SCI | SG'),
 Document(metadata={}, page_content='C100002 | LE001 | SP001 | Sunrise Logistics Pte Ltd | Sunrise Logistics Pte Ltd | 50 | Private Banking | Active | 18-09-2005 | Large | DEFAULT | 2010-09-02 | SCI | SG'),
 Document(metadata={}, page_content='C100003 | LE001 | SP001 | Sunrise Logistics Pte Ltd | Sunrise Logistics Pte Ltd | 30 | Business Clients | Active | 02-05-2016 | Small | DEFAULT | 2023-07-03 | SCI | SG'),
 Document(metadata={}, page_content='C100004 | LE001 | SP001 | SIMANGGANG SAWMILL COMPANY LTD SINGAPORE | SIMANGGANG SAWMILL COMPANY LTD SINGAPORE | K | Global corp

In [33]:
retriever = vectordb.as_retriever(search_kwargs={"k": 10})
res = retriever.get_relevant_documents("C100507") 
print("exact token retrieve:", len(res), res[:2])
res2 = retriever.get_relevant_documents("what is the customer name for client C100507")  # semantic
print("semantic retrieve:", len(res2), res2[:2])

exact token retrieve: 10 [Document(id='3b400011-38de-43d9-a318-ea1eba875472', metadata={}, page_content='C100977 | LE001 | SP001 | Pacific Exports Pte Ltd | Pacific Exports Pte Ltd | S | Own account | Active | 23-08-2015 | Small | DEFAULT | 2013-06-08 | SCI | SG'), Document(id='38978fb4-bc76-49ee-bfc1-527393ad9fda', metadata={}, page_content='C100647 | LE001 | SP001 | Cedar Technologies Pte Ltd | Cedar Technologies Pte Ltd | MM | Middle market | Active | 18-04-2007 | Medium | DEFAULT | 2012-12-21 | SCI | SG')]
semantic retrieve: 10 [Document(id='2df6dfdf-d855-4eea-91b6-b938f3deeb61', metadata={}, page_content='C100264 | LE001 | SP001 | Cedar Technologies Pte Ltd | Cedar Technologies Pte Ltd | 30 | Business Clients | Active | 09-01-2007 | Small | DEFAULT | 2016-08-14 | SCI | SG'), Document(id='a244fced-8989-48de-ac88-7257eb6847cf', metadata={}, page_content='C100608 | LE001 | SP001 | Apex Solutions Pte Ltd | Apex Solutions Pte Ltd | 30 | Business Clients | Active | 07-07-2010 | Medium |

In [53]:
path1="Data/ai_generated_data/upload_dataset_other_files/"

def load_all_data():
    """Load all CSV files into dataframes"""
    data = {}
    
    files = {
        'clients': 'client_master.csv',
        'core': 'core.csv',
        'collateral': 'collateral.csv',
        'custody': 'custody.csv',
        'facilities': 'facilities.csv',
        'loans': 'loans.csv',
        'fxd': 'fxd.csv',
        'funds': 'funds.csv',
        'trade_otp': 'trade_otp.csv',
        'trade_dtp': 'trade_dtp.csv'
    }
    
    
    for name, path in files.items():
        try:
            file_name=path1 + path
            print("Loading file:", file_name)
            data[name] = pd.read_csv(file_name)
        except FileNotFoundError:
            #st.warning(f"File not found: {path}, using empty DataFrame")
            data[name] = pd.DataFrame()
    
    print(data.values())
    return data

load_all_data()

Loading file: Data/ai_generated_data/upload_dataset_other_files/client_master.csv
Loading file: Data/ai_generated_data/upload_dataset_other_files/core.csv
Loading file: Data/ai_generated_data/upload_dataset_other_files/collateral.csv
Loading file: Data/ai_generated_data/upload_dataset_other_files/custody.csv
Loading file: Data/ai_generated_data/upload_dataset_other_files/facilities.csv
Loading file: Data/ai_generated_data/upload_dataset_other_files/loans.csv
Loading file: Data/ai_generated_data/upload_dataset_other_files/fxd.csv
Loading file: Data/ai_generated_data/upload_dataset_other_files/funds.csv
Loading file: Data/ai_generated_data/upload_dataset_other_files/trade_otp.csv
Loading file: Data/ai_generated_data/upload_dataset_other_files/trade_dtp.csv
dict_values([Empty DataFrame
Columns: []
Index: [],      le_id sub_prof_id client_id account_id account_type  \
0    LE001       SP001   C100654   AC100000         DEAL   
1    LE001       SP001   C100758   AC100001         DEAL   
2  

{'clients': Empty DataFrame
 Columns: []
 Index: [],
 'core':      le_id sub_prof_id client_id account_id account_type  \
 0    LE001       SP001   C100654   AC100000         DEAL   
 1    LE001       SP001   C100758   AC100001         DEAL   
 2    LE001       SP001   C100616   AC100002         DEAL   
 3    LE001       SP001   C100459   AC100003      ACCOUNT   
 4    LE001       SP001   C100284   AC100004         DEAL   
 ..     ...         ...       ...        ...          ...   
 995  LE001       SP001   C100487   AC100995         DEAL   
 996  LE001       SP001   C100631   AC100996      ACCOUNT   
 997  LE001       SP001   C100857   AC100997      ACCOUNT   
 998  LE001       SP001   C100438   AC100998      ACCOUNT   
 999  LE001       SP001   C100663   AC100999      ACCOUNT   
 
                               product product_type  product_code currency  \
 0                   FC FIXED DEPOSITS      DEPOSIT           614      AUD   
 1                    FC CALL DEPOSITS      DEPOS